<a href="https://colab.research.google.com/github/FatherNurt/FUNt-Cosmologiical-Model-of-All-Things/blob/main/C1_Log%E2%80%93%CE%A8_%E2%80%94_Extended_Run_with_Cont_Perturb_%26_Repair_%26_hHRT_w_relief_%26_2%E2%88%9A_style_constrained_transport_modulator_h%C2%B3%CF%80_detector_with_fibb_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n_steps = 6000
R = np.arange(n_steps)

# Base controller parameters
Kp = 0.65
Ki = 0.10
Kd = 0.18
constraint = 0.50

# === Fibonacci ramp + partial reset parameters ===
phi = (1 + np.sqrt(5)) / 2
base_noise = 0.04
max_noise  = 0.18
fib_partial_reset = phi - 1          # ≈ 0.618 (mild reset after each trigger)

def f_2sqrt_mod(i, R_t=2200.0, p=3.0):
    return (R_t ** p) / (i ** p + R_t ** p + 1e-8)

# Fibonacci correction constants
fib_integral_decay = phi - 1
fib_state_correction = 2 - phi

def run_fib_ramp_partial_reset():
    state = np.random.randn(1) * 0.1
    S = np.zeros(n_steps)
    psi = np.zeros(n_steps - 1)
    drive_history = np.zeros(n_steps)
    state_history = np.zeros(n_steps)
    integral = 0.0
    prev_error = 0.0

    h3pi_count = 0
    h3pi_times = []

    h3pi_window = 45
    h3pi_S_threshold = 0.17

    S_buffer = []
    effective_multiplier = 1.0          # starts at full stress, reduced after each trigger

    for i in range(n_steps):
        # Global Fibonacci-character ramp
        ramp = (i / n_steps) ** (phi - 1)
        global_noise = base_noise + (max_noise - base_noise) * ramp

        # Apply partial reset multiplier (Fibonacci factor after each h³π)
        noise_amplitude = global_noise * effective_multiplier

        base_drive = 0.6 + 0.25 * np.sin(2 * np.pi * i / 400)
        drive = base_drive + noise_amplitude * np.random.randn()

        error = drive - state[0]
        strain = abs(error)

        integral += error * 1.0
        derivative = (error - prev_error)

        pid_output = Kp * error + Ki * integral + Kd * derivative
        base_reorg = pid_output * (1 - constraint * abs(state[0]))
        transport_mod = f_2sqrt_mod(i)

        reorganization = base_reorg * transport_mod

        state = state + reorganization
        prev_error = error

        S[i] = strain
        drive_history[i] = drive
        state_history[i] = state[0]

        # === h³π detection ===
        S_buffer.append(S[i])
        if len(S_buffer) > h3pi_window:
            S_buffer.pop(0)

        if len(S_buffer) == h3pi_window:
            mean_S_window = np.mean(S_buffer)
            if mean_S_window > h3pi_S_threshold:
                # === Fibonacci-structured correction ===
                integral *= fib_integral_decay
                state = state * fib_state_correction

                # === Mild partial reset of effective stress (Fibonacci factor) ===
                effective_multiplier *= fib_partial_reset

                h3pi_count += 1
                h3pi_times.append(i)
                S_buffer = []

    dS = np.diff(S)
    dR = np.diff(R)
    psi = dS / dR

    psi_mean = np.mean(psi)
    psi_std = np.std(psi)
    growth = np.sum(psi > 0.01)
    decay = np.sum(psi < -0.01)
    stationary = np.sum((psi >= -0.01) & (psi <= 0.01))
    total = len(psi)
    mean_S_last = np.mean(S[-1000:])

    return {
        'S': S, 'psi': psi, 'state': state_history, 'drive': drive_history,
        'mean_psi': psi_mean, 'std_psi': psi_std,
        'growth_pct': 100 * growth / total,
        'decay_pct': 100 * decay / total,
        'stationary_pct': 100 * stationary / total,
        'mean_S_last1000': mean_S_last,
        'h3pi_count': h3pi_count,
        'h3pi_times': h3pi_times,
        'phi': phi,
        'final_multiplier': effective_multiplier
    }

result = run_fib_ramp_partial_reset()

print("=== Fibonacci Ramp + Partial Reset after h³π ===")
print(f"φ (golden ratio):        {result['phi']:.6f}")
print(f"Final stress multiplier: {result['final_multiplier']:.4f}")
print(f"Mean Ψ:                  {result['mean_psi']:.5f}")
print(f"Std Ψ:                   {result['std_psi']:.5f}")
print(f"Growth %:                {result['growth_pct']:.1f}%")
print(f"Decay %:                 {result['decay_pct']:.1f}%")
print(f"Quasi-stationary %:      {result['stationary_pct']:.1f}%")
print(f"Mean S (last 1000):      {result['mean_S_last1000']:.4f}")
print(f"h³π triggers:            {result['h3pi_count']}")
if result['h3pi_count'] > 0:
    print(f"First trigger at step: {result['h3pi_times'][0]}")
    print(f"Last trigger at step:  {result['h3pi_times'][-1]}")

=== Fibonacci Ramp + Partial Reset after h³π ===
φ (golden ratio):        1.618034
Final stress multiplier: 0.6180
Mean Ψ:                  -0.00007
Std Ψ:                   0.10189
Growth %:                45.3%
Decay %:                 45.2%
Quasi-stationary %:      9.6%
Mean S (last 1000):      0.0874
h³π triggers:            1
First trigger at step: 4235
Last trigger at step:  4235
